# argentina.salud — Pruebas interactivas

Recorrido paso a paso del módulo `argentina.salud`.

Funciones simples para normalizar variables frecuentes en datos administrativos de salud (sexo, tipo de documento, matrícula profesional, grupos etarios, edad). Solo stdlib — sin curvas OMS, sin z-scores, sin pandas, sin APIs.

## 1. Setup e imports

In [1]:
import argentina as arg

print(f"argentina v{arg.__version__}")

argentina v0.0.16


## 2. normalizar_sexo

Mapea variantes a 3 valores canónicos: `M`, `F`, `X`.

In [2]:
for v in ["femenino", "mujer", "F", "masculino", "varón", "hombre", "M", "no binario", "otro", "X"]:
    print(f"{v!r:15} → {arg.salud.normalizar_sexo(v)!r}")

'femenino'      → 'F'
'mujer'         → 'F'
'F'             → 'F'
'masculino'     → 'M'
'varón'         → 'M'
'hombre'        → 'M'
'M'             → 'M'
'no binario'    → 'X'
'otro'          → 'X'
'X'             → 'X'


In [3]:
# Sin match → None (no inventa)
print(arg.salud.normalizar_sexo("otroX"))
print(arg.salud.normalizar_sexo(None))
print(arg.salud.normalizar_sexo(""))

None
None
None


## 3. normalizar_tipo_documento

DNI, LC, LE, Pasaporte, CI.

In [4]:
for v in ["dni", "DNI", "Documento Nacional de Identidad", "LC", "libreta cívica", "pasaporte", "PAS", "CI", "cédula", "otro"]:
    print(f"{v!r:38} → {arg.salud.normalizar_tipo_documento(v)!r}")

'dni'                                  → 'DNI'
'DNI'                                  → 'DNI'
'Documento Nacional de Identidad'      → 'DNI'
'LC'                                   → 'LC'
'libreta cívica'                       → 'LC'
'pasaporte'                            → 'PASAPORTE'
'PAS'                                  → 'PASAPORTE'
'CI'                                   → 'CI'
'cédula'                               → 'CI'
'otro'                                 → None


## 4. limpiar_matricula

Saca espacios, puntos y signos. Conserva letras (mayúsculas) y dígitos.

In [5]:
for v in ["M.P. 12345", "mn 67890", "  MP-12345  ", 12345, "--", None]:
    print(f"{v!r:18} → {arg.salud.limpiar_matricula(v)!r}")

'M.P. 12345'       → 'MP12345'
'mn 67890'         → 'MN67890'
'  MP-12345  '     → 'MP12345'
12345              → '12345'
'--'               → None
None               → None


## 5. grupo_etario

Franjas estándar: `0` (menores de 1), `1-4`, `5-9`, …, `55-64`, `65+`.

In [6]:
for edad in [0, 0.5, 1, 4, 5, 9, 14, 19, 24, 34, 44, 54, 64, 65, 70, 100]:
    print(f"{edad:>5} → {arg.salud.grupo_etario(edad)}")

    0 → 0
  0.5 → 0
    1 → 1-4
    4 → 1-4
    5 → 5-9
    9 → 5-9
   14 → 10-14
   19 → 15-19
   24 → 20-24
   34 → 25-34
   44 → 35-44
   54 → 45-54
   64 → 55-64
   65 → 65+
   70 → 65+
  100 → 65+


In [7]:
# Inválidos → None
print(arg.salud.grupo_etario(-1))
print(arg.salud.grupo_etario(None))
print(arg.salud.grupo_etario("abc"))

None
None
None


## 6. edad_en_anios

Acepta strings ISO (`'YYYY-MM-DD'`), `date` o `datetime`. Respeta si ya cumplió años en la fecha de referencia.

In [8]:
# Caso típico
arg.salud.edad_en_anios("2015-05-10", "2026-05-12")

11

In [9]:
# Antes del cumple del año actual → resta 1
arg.salud.edad_en_anios("2015-12-10", "2026-05-12")

10

In [10]:
# Acepta date / datetime
from datetime import date, datetime

print(arg.salud.edad_en_anios(date(2000, 1, 1), date(2026, 5, 12)))
print(arg.salud.edad_en_anios(datetime(2000, 1, 1, 12, 0), "2026-05-12"))

26
26


In [11]:
# Por defecto compara contra hoy
arg.salud.edad_en_anios("2000-01-01")

26

In [12]:
# Casos borde → None
print(arg.salud.edad_en_anios("fecha mala", "2026-05-12"))
print(arg.salud.edad_en_anios(None, "2026-05-12"))
print(arg.salud.edad_en_anios("2030-01-01", "2026-05-12"))   # nacimiento futuro

None
None
None


## 7. Combinando todo

Pipeline típico: una fila cruda de un padrón sanitario con sexo, tipo de doc, fecha de nacimiento y matrícula del médico tratante.

In [13]:
registros = [
    {"sexo": "Femenino",  "tipo_doc": "DNI",       "nacimiento": "1985-03-22", "matricula": "M.P. 12345"},
    {"sexo": "varón",     "tipo_doc": "pasaporte", "nacimiento": "2020-11-08", "matricula": "mn-987"},
    {"sexo": "otro",      "tipo_doc": "cédula",    "nacimiento": "1955-01-10", "matricula": None},
    {"sexo": "",          "tipo_doc": "otro",      "nacimiento": "fecha mala", "matricula": "--"},
]

REF = "2026-05-12"
for r in registros:
    edad = arg.salud.edad_en_anios(r["nacimiento"], REF)
    print({
        "sexo":      arg.salud.normalizar_sexo(r["sexo"]),
        "tipo_doc":  arg.salud.normalizar_tipo_documento(r["tipo_doc"]),
        "edad":      edad,
        "grupo":     arg.salud.grupo_etario(edad),
        "matricula": arg.salud.limpiar_matricula(r["matricula"]),
    })

{'sexo': 'F', 'tipo_doc': 'DNI', 'edad': 41, 'grupo': '35-44', 'matricula': 'MP12345'}
{'sexo': 'M', 'tipo_doc': 'PASAPORTE', 'edad': 5, 'grupo': '5-9', 'matricula': 'MN987'}
{'sexo': 'X', 'tipo_doc': 'CI', 'edad': 71, 'grupo': '65+', 'matricula': None}
{'sexo': None, 'tipo_doc': None, 'edad': None, 'grupo': None, 'matricula': None}


## 8. Tests automáticos

```bash
cd /Users/tobiasyatche/argentina
pytest tests/test_salud.py -v
```

## Notas sueltas / TODOs

- Solo stdlib (`re`, `unicodedata`, `datetime`). Sin pandas, sin curvas OMS, sin z-scores, sin APIs externas.
- `normalizar_sexo` devuelve `M`/`F`/`X`. Si en algún momento se necesita un cuarto valor (`I` para indeterminado en RENAPER, por ejemplo) hay que extender el mapping.
- Las franjas etarias son las que usa la **DEIS** para mortalidad (0, 1-4, 5-9, …, 65+). Si necesitás franjas distintas (decenales puras, o franjas pediátricas más finas) conviene una función aparte.
- Antropometría (talla, peso, IMC, percentilos OMS) queda explícitamente fuera del scope.